# Sound Processing Lab - Part #3

In this notebook you'll learn:
- Sound dataset creation and normalization
- Build a pre and post feature extraction Neural Network implementation and optimize it!

In [ ]:
# Imports and configuration
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

## A Four-step Journey
<ol type="a">
    <li>Audio Intake, Truncation & Amplitude Normalization</li>
    <li>Feature Extraction & The Visual Cluster Map (PCA)</li>
    <li>Training the Network & Drawing the Dynamic Brain</li>
    <li>The Live Inference Arena</li>
</ol>

### A. Audio Intake, Truncation & Amplitude Normalization
This cell automatically scans your three folders, trims leading/trailing dead space, forces every clip to be exactly 5.0 seconds long, and scales the volume so the network isn't fooled by how loud a recording is.

In [ ]:
# ----------------------------------------------------------------------------------------
# A. Audio Intake, Truncation & Amplitude Normalization
# ----------------------------------------------------------------------------------------
# 1. Setup paths and parameters
DATA_DIR = '..\\sounds'  # Assumes subfolders: sounds/cat, sounds/dog, sounds/elephant
classes = ['dog', 'elephant'] # we leave the rest of the animals for later

sampling_rate = 22050  
target_duration = 5.0
target_length = int(sampling_rate * target_duration)  # 110,250 samples

X_raw_audio = []
y = []
file_info = []

# Diagnostics: Storage to capture the lifecycle of our "Classroom Mascot" file
mascot_pipeline = {}
mascot_captured = False

print("📂 Processing Classroom Dataset (Dogs vs. Elephants)...")

for class_index, class_name in enumerate(classes):
    class_folder = os.path.join(DATA_DIR, class_name)
    if not os.path.exists(class_folder):
        print(f"⚠️ Folder missing: {class_folder}. Please create it!")
        continue
        
    files = [f for f in os.listdir(class_folder) if f.endswith('.mp3') or f.endswith('.wav')]
    print(f" -> Found {len(files)} files in '{class_name}' folder.")
    
    for file_name in files:
        file_path = os.path.join(class_folder, file_name)
        try:
            # --- STAGE 1: RAW INTAKE ---
            signal, sr = librosa.load(file_path, sr=sampling_rate)
            
            # If this is the first file, capture its raw state for the class display
            if not mascot_captured:
                mascot_pipeline['filename'] = f"{class_name.upper()}: {file_name}"
                mascot_pipeline['sr'] = sr
                mascot_pipeline['1_raw'] = signal.copy()
                is_mascot = True
                mascot_captured = True
            else:
                is_mascot = False
            
            # --- STAGE 2: TOP & TAIL TRIM ---
            # Removes absolute dead background hiss/silence below 25 decibels
            signal_trimmed, _ = librosa.effects.trim(signal, top_db=25)
            if is_mascot: mascot_pipeline['2_trimmed'] = signal_trimmed.copy()
            
            # --- STAGE 3: LENGTH ENFORCEMENT ---
            # Forces the clip to be exactly 5.0 seconds without introducing dead zero-silence
            if len(signal_trimmed) >= target_length:
                signal_fixed = signal_trimmed[:target_length]
            else:
                signal_fixed = np.pad(signal_trimmed, (0, target_length - len(signal_trimmed)), 'reflect')
            if is_mascot: mascot_pipeline['3_fixed_length'] = signal_fixed.copy()
            
            # --- STAGE 4: AMPLITUDE NORMALIZATION ---
            # Standardizes the peak volume scale to 1.0 so microphone distance doesn't confuse the AI
            signal_norm = librosa.util.normalize(signal_fixed)
            if is_mascot: mascot_pipeline['4_normalized'] = signal_norm.copy()
            
            # Append final clean data to training arrays
            X_raw_audio.append(signal_norm)
            y.append(class_index)
            file_info.append(f"{class_name.upper()}: {file_name}")
            
        except Exception as e:
            print(f" ❌ Error processing {file_name}: {e}")

X_raw_audio = np.array(X_raw_audio)
y = np.array(y)

print(f"\n✅ Data Cleaned! Final Matrix Shape: {X_raw_audio.shape}")
print(f"🎨 Rendering Data Hygiene Timeline for Mascot File: {mascot_pipeline['filename']}...\n")

# --- DRAW THE WAVEFORM EVOLUTION TIMELINE ---
fig, axs = plt.subplots(4, 1, figsize=(13, 10), sharex=False)
sr_mascot = mascot_pipeline['sr']

# Panel 1: Raw Waveform
librosa.display.waveshow(mascot_pipeline['1_raw'], sr=sr_mascot, ax=axs[0], color='gray', alpha=0.7)
axs[0].set_title(f"STAGE 1: Raw Audio (As Found on Disk) — Length: {len(mascot_pipeline['1_raw'])/sr_mascot:.2f}s", fontsize=11, fontweight='bold')
axs[0].set_ylabel("Amplitude")

# Panel 2: Trimmed Waveform
librosa.display.waveshow(mascot_pipeline['2_trimmed'], sr=sr_mascot, ax=axs[1], color='darkorange')
axs[1].set_title(f"STAGE 2: Trimming Dead Air (Removes leading/trailing noise floors) — Length: {len(mascot_pipeline['2_trimmed'])/sr_mascot:.2f}s", fontsize=11, fontweight='bold')
axs[1].set_ylabel("Amplitude")

# Panel 3: Fixed Length Waveform
librosa.display.waveshow(mascot_pipeline['3_fixed_length'], sr=sr_mascot, ax=axs[2], color='royalblue')
axs[2].set_title(f"STAGE 3: Reflection Padding/Truncating (Enforcing precise baseline target) — Length: {len(mascot_pipeline['3_fixed_length'])/sr_mascot:.2f}s", fontsize=11, fontweight='bold')
axs[2].set_ylabel("Amplitude")

# Panel 4: Normalized Waveform
librosa.display.waveshow(mascot_pipeline['4_normalized'], sr=sr_mascot, ax=axs[3], color='forestgreen')
axs[3].set_title("STAGE 4: Amplitude Normalization (Scaling max peak volume to 1.0 boundary)", fontsize=11, fontweight='bold')
axs[3].set_ylabel("Normalized Amp")
axs[3].set_xlabel("Time (Seconds)")

# Clean up layout display
for ax in axs:
    ax.label_outer()
    ax.grid(True, linestyle=':', alpha=0.6)

plt.suptitle(f"The Data Preprocessing Pipeline in Action\nMascot Diagnostics: {mascot_pipeline['filename']}", fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# --- NEW: VISUAL DATASET BLUEPRINT FOR THE CLASSROOM ---
print("\n" + "="*80)
print("🗺️  THE SUPERVISED LEARNING SPREADSHEET MAP (Ground Truth Answer Key)")
print("="*80)
print(f"{'Row Index':<12} | {'Feature Columns (X)':<28} | {'Target Label Column (y)':<25}")
print("-"*80)

# Print every row in the dataset to show the explicit pairing
for idx in range(len(X_raw_audio)):
    feature_snippet = f"[{X_raw_audio[idx][0]:.3f}, {X_raw_audio[idx][1]:.3f}, ..., {X_raw_audio[idx][-1]:.3f}]"
    class_string = f"{y[idx]} ({classes[y[idx]].upper()})"
    print(f"Row #{idx:<9} | {feature_snippet:<28} | {class_string:<25}")

print("="*80)
print("💡 TEACHING POINT: Every single row has an 'X' (Features) AND a 'y' (The Answer).")
print("   Without this 'y' column, a Neural Network cannot calculate its loss or learn!")
print("="*80)

### From the cell above:  

**Neural Networks CANNOT process text, they're uncapable of doing it as such; so labels must be translated into numerical labels. This process is called: "Label Encoding" or "Integer Encoding".**

## B. Feature Extraction & The Visual Cluster Map (PCA)
Before training the network, we extract the MFCCs. Then, we use PCA to flatten those massive feature grids into a 2D coordinate system so students can see their animals clump into distinct zones before the AI even touches them.

In [ ]:
# ----------------------------------------------------------------------------------------
# B. Feature Extraction & The Visual Cluster Map (PCA)
# ----------------------------------------------------------------------------------------
print("🧬 Extracting Acoustic Fingerprints (MFCCs)...")
X_mfcc = []

for signal in X_raw_audio:
    mfcc = librosa.feature.mfcc(y=signal, sr=sampling_rate, n_mfcc=20)
    X_mfcc.append(mfcc.flatten())

X_mfcc = np.array(X_mfcc)

# Scale features for the Neural Network
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_mfcc)

# Run PCA to squash features into 2 dimensions
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Plot the Pre-Class Cluster Map (Fixed colors/markers for 2 classes)
plt.figure(figsize=(9, 6))
colors = ['crimson', 'darkorange']
markers = ['s', '^']

for class_index, class_name in enumerate(classes):
    mask = (y == class_index)
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], 
                c=colors[class_index], marker=markers[class_index], 
                s=150, label=class_name.upper(), edgecolors='k', alpha=0.8)

plt.title("Acoustic Separation Map (PCA Space)", fontsize=14, fontweight='bold')
plt.xlabel("Principal Component 1 (Major Texture Shift)", fontsize=11)
plt.ylabel("Principal Component 2 (Minor Texture Shift)", fontsize=11)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--')
plt.show()

print(f"💡 Visual Check: Are the {classes[0].upper()}s and {classes[1].upper()}s completely split into groups?")

## C. Training the Network & Drawing the Dynamic Brain
Here we train the network. Once trained, this code bypasses abstract grids and manually draws the classic node-link diagram. It displays the input layer limits, all hidden processing units, and the 3 final animal outputs, color-coding the connection lines based on the true trained weights.

In [ ]:
# ----------------------------------------------------------------------------------------
# C. Training the Network & Drawing the Dynamic Brain
# ----------------------------------------------------------------------------------------
# 1. Initialize and train a Neural Network with 8 hidden neurons
hidden_nodes = 8
mlp = MLPClassifier(hidden_layer_sizes=(hidden_nodes,), max_iter=1000, random_state=42, learning_rate_init=0.01)
mlp.fit(X_scaled, y)

weights = mlp.coefs_[0] 
n_inputs = weights.shape[0]

# 2. GRAPHICALLY DRAW THE NEURAL NETWORK
fig, ax = plt.subplots(figsize=(12, 7))
ax.axis('off')

x_in, x_hid, x_out = 0.1, 0.5, 0.9

# Draw Input Layer Indicators
visible_inputs = 6
input_y_positions = np.linspace(0.1, 0.9, visible_inputs)
for idx, y_pos in enumerate(input_y_positions):
    label = f"MFCC_{idx*35}" if idx < visible_inputs-1 else f"MFCC_{n_inputs}"
    ax.scatter(x_in, y_pos, s=400, facecolors='white', edgecolors='black', linewidth=2, zorder=3)
    ax.text(x_in - 0.08, y_pos, label, va='center', ha='right', fontsize=9, fontweight='bold')
ax.text(x_in, 0.5, "•\n•\n•", va='center', ha='center', fontsize=20, fontweight='bold', color='gray')

# Draw Hidden Layer Nodes
hidden_y_positions = np.linspace(0.15, 0.85, hidden_nodes)
for idx, y_pos in enumerate(hidden_y_positions):
    ax.scatter(x_hid, y_pos, s=500, facecolors='lightgray', edgecolors='purple', linewidth=2, zorder=3)
    ax.text(x_hid, y_pos, f"H_{idx}", va='center', ha='center', fontsize=9, fontweight='bold')

# Draw Output Layer Nodes (Fixed font warning by removing the emoji)
output_y_positions = np.linspace(0.35, 0.65, 2)
for idx, y_pos in enumerate(output_y_positions):
    ax.scatter(x_out, y_pos, s=800, facecolors='white', edgecolors='green', linewidth=2, zorder=3)
    ax.text(x_out + 0.03, y_pos, f"Class: {classes[idx].upper()}", va='center', ha='left', fontsize=12, fontweight='bold')

# Draw Input-to-Hidden Synapse Lines
vmax = np.max(np.abs(weights))
for i, in_y in enumerate(input_y_positions):
    w_idx = int(i * (n_inputs / visible_inputs))
    if w_idx >= n_inputs: w_idx = n_inputs - 1
    for h, hid_y in enumerate(hidden_y_positions):
        weight_val = weights[w_idx, h]
        color = 'crimson' if weight_val > 0 else 'royalblue'
        linewidth = (np.abs(weight_val) / vmax) * 3.5
        ax.plot([x_in, x_hid], [in_y, hid_y], color=color, alpha=0.4, linewidth=linewidth, zorder=1)

# Draw Hidden-to-Output Connections (Dynamically adjusts for binary vs multi-class shapes)
hidden_to_out_weights = mlp.coefs_[1]
max_out_w = np.max(np.abs(hidden_to_out_weights))

for h, hid_y in enumerate(hidden_y_positions):
    for o, out_y in enumerate(output_y_positions):
        # Handle scikit-learn's single-neuron binary weight array mapping
        if hidden_to_out_weights.shape[1] == 1:
            raw_w = hidden_to_out_weights[h, 0]
            # Positive weight favors Elephant (Class 1), negative weight favors Dog (Class 0)
            weight_val = raw_w if o == 1 else -raw_w
        else:
            weight_val = hidden_to_out_weights[h, o]
            
        color = 'crimson' if weight_val > 0 else 'royalblue'
        linewidth = (np.abs(weight_val) / max_out_w) * 4
        ax.plot([x_hid, x_out], [hid_y, out_y], color=color, alpha=0.6, linewidth=linewidth, zorder=1)

plt.title(f"Trained Bio-Acoustic Brain Map\nInput Array ({n_inputs} Features) ➔ Hidden Hub ({hidden_nodes} Neurons) ➔ 2 Class Targets", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## C-2. Training & Plotting the PCA-Optimized Network
By training the network directly on the 2 PCA (Principal Component Analysis) components generated in setp B, the input layer drops from 4,320 features down to just 2 input nodes.  
In Machine Learning, this technique is used for dimensionality reduction. It simplifies complex datasets by reducing the number of variables (features) while preserving the most critical information and patterns.

In [ ]:
# 1. Train the Network on the PCA-Compressed Features (From Cell 2)
hidden_nodes = 8
mlp = MLPClassifier(hidden_layer_sizes=(hidden_nodes,), max_iter=1000, random_state=42, learning_rate_init=0.01)
mlp.fit(X_pca, y)  # Enters the brain using only 2 dimensions!

weights = mlp.coefs_[0] 
n_inputs = weights.shape[0]

# 2. GRAPHICALLY DRAW THE OPTIMIZED NEURAL NETWORK
fig, ax = plt.subplots(figsize=(11, 6))
ax.axis('off')

x_in, x_hid, x_out = 0.1, 0.5, 0.9

# Draw Input Layer (Exactly 2 PCA Nodes—No Ellipsis Needed!)
input_y_positions = [0.4, 0.6]
for idx, y_pos in enumerate(input_y_positions):
    label = f"Input: PCA_{idx+1}"
    ax.scatter(x_in, y_pos, s=500, facecolors='white', edgecolors='black', linewidth=2, zorder=3)
    ax.text(x_in - 0.04, y_pos, label, va='center', ha='right', fontsize=10, fontweight='bold')

# Draw Hidden Layer Nodes (8 Processing Hubs)
hidden_y_positions = np.linspace(0.15, 0.85, hidden_nodes)
for idx, y_pos in enumerate(hidden_y_positions):
    ax.scatter(x_hid, y_pos, s=500, facecolors='lightgray', edgecolors='purple', linewidth=2, zorder=3)
    ax.text(x_hid, y_pos, f"H_{idx}", va='center', ha='center', fontsize=9, fontweight='bold')

# Draw Output Layer Nodes (2 Targets)
output_y_positions = [0.35, 0.65]
for idx, y_pos in enumerate(output_y_positions):
    ax.scatter(x_out, y_pos, s=800, facecolors='white', edgecolors='green', linewidth=2, zorder=3)
    ax.text(x_out + 0.03, y_pos, f"Class: {classes[idx].upper()}", va='center', ha='left', fontsize=12, fontweight='bold')

# Draw Input-to-Hidden Synapses
vmax = np.max(np.abs(weights))
for i, in_y in enumerate(input_y_positions):
    for h, hid_y in enumerate(hidden_y_positions):
        weight_val = weights[i, h]
        color = 'crimson' if weight_val > 0 else 'royalblue'
        linewidth = (np.abs(weight_val) / vmax) * 4.0
        ax.plot([x_in, x_hid], [in_y, hid_y], color=color, alpha=0.5, linewidth=linewidth, zorder=1)

# Draw Hidden-to-Output Synapses
hidden_to_out_weights = mlp.coefs_[1]
max_out_w = np.max(np.abs(hidden_to_out_weights))

for h, hid_y in enumerate(hidden_y_positions):
    for o, out_y in enumerate(output_y_positions):
        if hidden_to_out_weights.shape[1] == 1:
            raw_w = hidden_to_out_weights[h, 0]
            weight_val = raw_w if o == 1 else -raw_w
        else:
            weight_val = hidden_to_out_weights[h, o]
            
        color = 'crimson' if weight_val > 0 else 'royalblue'
        linewidth = (np.abs(weight_val) / max_out_w) * 4.5
        ax.plot([x_hid, x_out], [hid_y, out_y], color=color, alpha=0.6, linewidth=linewidth, zorder=1)

plt.title("Optimized Bio-Acoustic Brain Map\nInput Space Compressed via PCA (2 Features) ➔ Hidden Hub (8 Neurons) ➔ 2 Class Targets", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## D. The Live Inference Arena
This is the interactive playground. You place completely unseen verification files into the main directory (like a new bark or a cat's meow), run this cell, and watch the model calculate its classification scores.  
You can find a sheep and horse sounds to try some more.

In [ ]:
# ----------------------------------------------------------------------------------------
# D. The Live Inference Arena
# ----------------------------------------------------------------------------------------
# 🚀 TEST ARENA: Run your test files through the updated PCA brain pipeline
test_filename = r'..\sounds\cat\cat_meow_1.mp3' 

if not os.path.exists(test_filename):
    print(f"⚠️ Test file '{test_filename}' not found. Check your relative path slashes!")
else:
    # 1. Run the standardized raw audio cleaning pipeline
    sig, sr = librosa.load(test_filename, sr=sampling_rate)
    sig, _ = librosa.effects.trim(sig, top_db=25)
    if len(sig) >= target_length:
        sig = sig[:target_length]
    else:
        sig = np.pad(sig, (0, target_length - len(sig)), 'reflect')
    sig = librosa.util.normalize(sig)
    
    # 2. Step A: Extract raw 4,320 MFCC structural values
    feat_mfcc = librosa.feature.mfcc(y=sig, sr=sampling_rate, n_mfcc=20).flatten().reshape(1, -1)
    
    # 3. Step B: Apply training standardization scale
    feat_scaled = scaler.transform(feat_mfcc)
    
    # 4. Step C: Compress the structural values down to the 2 core spatial coordinates
    feat_pca = pca.transform(feat_scaled)
    
    # 5. Query the optimized PCA network
    predicted_class = mlp.predict(feat_pca)[0]
    probabilities = mlp.predict_proba(feat_pca)[0]
    
    print("--- 🧠 LIVE ARTIFICIAL INTELLIGENCE INFERENCE VERDICT ---")
    print(f"🔊 Loaded File: {test_filename}")
    print(f"🎯 Forced Output Choice: {classes[predicted_class].upper()}")
    print("\n📊 Neural Probability Profile:")
    for idx, class_name in enumerate(classes):
        print(f"   ➔ Probability of {class_name.upper()}: {probabilities[idx]*100:.2f}%")

## What have we learned?

Let's discuss in class.



Got more time and want to continue playing? Cool!
* We could have used other non-supervised techniques for dataset classification, like k-means (clustering)
* Dimensionality can be a curse, the more complex the model the hardest it is and much higher the cost.
    * **Food for thought:** As you saw on the first cell when building the dataset, think about the size of the dataset in columns when you send someone a voice message over Whatsapp. Do the math, how many columns at 44.100 Hz sampling frequency?
* When using supervised learning, remember the amount of columns in the dataset become the amount of neurons in the entry layer of the Neural Network:
    * The Feature Columns (The Input Neurons): Every column that contains a piece of data we use to describe the sound (like our MFCC values or PCA coordinates) maps directly to exactly one neuron in the entry layer.
    * The Label Column (The Output Neurons): The very last column—the "Target" or "Class" column that holds the answer (0 for dog, 1 for elephant)—does not go into the input layer. Instead, it tells the network what the output layer should look like.

    * Re-check the code above and think:
        * The Raw Dataset: When we flattened the raw sound features, our dataset matrix had 4,320 feature columns per row. Because of that, the entry layer of our first neural network had exactly 4,320 neurons.
        * The PCA-Optimized Dataset: When we ran PCA, we compressed those 4,320 columns down to just 2 feature columns (Principal Component 1 and Principal Component 2). Immediately, the entry layer of our network shrank to exactly 2 neurons.
    * Bottom line: The width of your input data dictates the size of the brain's front door.


### How PCA works
Instead of just deleting features, PCA mathematically combines them into entirely new, uncorrelated variables called principal components.  
* **Maximizing Variance:** It ranks these components so that the first principal component (PC₁) captures the maximum amount of variation in the data.
* **Remaining Variation:** Each subsequent component captures the maximum remaining variation while remaining strictly uncorrelated (orthogonal) to the previous ones.
* **Dropping Dimensions:** You can drop the lower-ranked components (which usually carry mostly noise) without losing significant structural information.

**Why is it used?**  
PCA is generally applied right before training a machine learning model to tackle several common issues:
* **Overcoming the "Curse of Dimensionality":** Models can struggle to generalize when there are too many features. PCA shrinks the feature space, improving computational speed and model performance.
* **Fixing Multicollinearity:** It eliminates issues where original features are highly correlated with one another, making causal modeling more stable.
* **Data Visualization:** It allows data scientists to project multi-dimensional data (e.g., 50 columns) into 2D or 3D scatter plots.

**When to use it?**  
PCA is an unsupervised learning method (meaning it only looks at the features, not the target labels). It is highly useful when working with high-dimensional datasets like image pixels, genomics, or natural language processing

### More to learn in non-supervised classification techniques: K-Means
It's a unsupervised Machine Learning algorithm used to group unlabeled data into "K" distinct clusters of information based on similarity.

In [ ]:
print("🤖 Running K-Means (Unsupervised Clustering)...")

# 1. Initialize K-Means to find exactly 2 organic groups in our PCA coordinates
# Note: K-Means does NOT look at the target array 'y' at all!
k_means = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels = k_means.fit_predict(X_pca)

# Extract the calculated center points (centroids) of the two discovered shapes
centroids = k_means.cluster_centers_

# 2. Plot Side-by-Side Comparison: Real Life vs. Computer Blind Grouping
fig, axs = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)

colors_real = ['crimson', 'darkorange']
markers_real = ['s', '^']
colors_cluster = ['royalblue', 'forestgreen']

# Left Plot: The Reality (Ground Truth Labels)
for class_index, class_name in enumerate(classes):
    mask = (y == class_index)
    axs[0].scatter(X_pca[mask, 0], X_pca[mask, 1], 
                   c=colors_real[class_index], marker=markers_real[class_index], 
                   s=150, label=class_name.upper(), edgecolors='k', alpha=0.8)
axs[0].set_title("1. Ground Truth (Human Labels Specified)", fontsize=13, fontweight='bold')
axs[0].set_xlabel("Principal Component 1")
axs[0].set_ylabel("Principal Component 2")
axs[0].legend(fontsize=11)
axs[0].grid(True, linestyle='--')

# Right Plot: The Blind Grouping (K-Means Discovered Clusters)
for cluster_idx in range(2):
    mask = (cluster_labels == cluster_idx)
    axs[1].scatter(X_pca[mask, 0], X_pca[mask, 1], 
                   c=colors_cluster[cluster_idx], marker='o', 
                   s=150, label=f"CLUSTER {cluster_idx}", edgecolors='k', alpha=0.8)

# Overlay the mathematical gravity wells (Centroids)
axs[1].scatter(centroids[:, 0], centroids[:, 1], 
               c='yellow', marker='X', s=300, label="CENTROIDS", edgecolors='black', linewidth=2)

axs[1].set_title("2. K-Means Clusters (Discovered Purely by Distance)", fontsize=13, fontweight='bold')
axs[1].set_xlabel("Principal Component 1")
axs[1].legend(fontsize=11)
axs[1].grid(True, linestyle='--')

plt.suptitle("Supervised Versus Unsupervised Audio Classification Profiles", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# 3. Print a diagnostic insight for the classroom
print("💡 CLASSROOM OBSERVATION CONTEXT:")
print("Look closely at the two graphs. K-Means grouped the files without knowing which was a dog or an elephant.")
print("If the shapes on the left match the color zones on the right, the physics of these sounds are so naturally different that labels are barely even required to sort them!")

### Two key facts on k-means:
* The Arbitrary Label Phenomenon: K-means might assign all the Dogs to Cluster 1 and all the Elephants to Cluster 0. Explain to them that the computer doesn't know what a "dog" or a "bark" is—it just knows that the data points in Cluster 1 share a high acoustic similarity, and the data points in Cluster 0 share a different one.

* The Centroids ("Gravity Wells"): The yellow X markers represent the mathematical mathematical average of those audio signatures. The closer a sound is to a centroid, the more "prototypical" it is of that group's texture.

**Common Applications**
* **Customer Segmentation:** Grouping buyers into profiles to target marketing efforts.
* **Image Compression:** Reducing the number of distinct colors in an image by grouping similar shades.
* **Anomaly Detection:** Flagging data points that are excessively far from any cluster centroid as outliers.

**Pros and Cons**
* **Pros:** It is simple to understand, scales well to large datasets, and is computationally efficient.
* **Cons:** You must define the number of clusters (K) in advance, and the algorithm struggles with clusters that are not roughly circular, density-based, or have overlapping data.


## Bonus track of the chapter - The Elbow Method
The Elbow Method is the standard data science tactic for answering the question: "How do we know how many groups are actually hidden in this data without looking at the labels?" It works by running K-Means multiple times, each time increasing the number of clusters ($k$), and measuring the Inertia (also known as the Within-Cluster Sum of Squares).  
Inertia calculates how far away the data points are from their assigned cluster centers. As you add more clusters, the inertia will always drop because the centers get closer to the points.The secret is finding the "elbow joint" on the graph—the exact point where adding another cluster gives you diminishing returns and the line flattens out.

In [ ]:
print("📊 Calculating cluster inertia across multiple configurations...")

# 1. Test a range of possible cluster sizes (k=1 through k=6)
# Given the small dataset size, testing up to 5 or 6 is perfect for demonstration
k_range = range(1, 6)
inertia_values = []

for k in k_range:
    # Initialize K-Means for the current target cluster count
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_pca)
    
    # Grab the inertia (total squared distance from points to their centroids)
    inertia_values.append(km.inertia_)

# 2. Plot the Elbow Curve
plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia_values, marker='o', linestyle='-', color='purple', linewidth=2.5, markersize=9)

# Highlight the "Elbow Point" explicitly for the students
plt.axvline(x=2, color='crimson', linestyle='--', alpha=0.7, label='The Elbow Point (k=2)')

plt.title("The Elbow Method Curve (Acoustic Variance Analysis)", fontsize=13, fontweight='bold')
plt.xlabel("Number of Clusters (k)", fontsize=11)
plt.ylabel("Cluster Inertia (Internal Dataset Strain)", fontsize=11)
plt.xticks(k_range)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--')
plt.show()

# 3. Print the analytical takeaway
print("💡 CLASSROOM DISCUSSION GUIDE:")
print(f" -> At k=1: The inertia is at its maximum because the computer forces all dogs and elephants into one massive group.")
print(f" -> At k=2: The inertia drops drastically because the data cleanly splits into two native pools.")
print(f" -> After k=2: The line levels off. Splitting the data into 3 or 4 groups doesn't explain the physics of the sound any better.")

### Another twist to the nut - The Rubber Band Analogy
* **The Rubber Band Analogy:** Imagine stretching a single giant rubber band around every single audio point on their scatter plot (k = 1). The band is under an immense amount of physical tension (High Inertia).
* **Snapping into Groups:** If you let the computer use two rubber bands (k = 2), one wraps around the dogs and one wraps around the elephants. The tension drops instantly because the bands don't have to stretch across that giant empty vacuum between the species.
* **Over-Splitting:** If you tell the computer to use three rubber bands (k = 3), it has to awkwardly slice either the dogs or elephants in half. The tension barely drops at all, which creates that flat line after the elbow.
